In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# ============================================================================
# LOAD DATA
# ============================================================================

print("Loading data...")
df = pd.read_parquet("../Data/Parquet/New_estchlsummed.parquet")

print(f"Original dataset: {len(df)} rows")

# ============================================================================
# PREPARE DATA
# ============================================================================

# Features and target
X = df[["ESTCHL_SUMMED", "K_PAR_SLOPE", "XMISS_SUMMED"]].copy()
y = df["Secchi"].copy()

# Remove rows with missing values in features or target
mask = X.notna().all(axis=1) & y.notna() & (y > 0)
X_clean = X[mask]
y_clean = y[mask]

# *** LOG TRANSFORM TARGET (same as polynomial model) ***
y_log = np.log(y_clean)

print(f"Clean dataset: {len(X_clean)} rows")
print(f"Features: {list(X_clean.columns)}")

# ============================================================================
# TRAIN RANDOM FOREST ON LOG SCALE
# ============================================================================

print("\n" + "="*80)
print("TRAINING RANDOM FOREST ON LOG(SECCHI) - 100% DATA")
print("="*80)

# Initialize Random Forest
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_depth=10,
    min_samples_split=5
)

# 5-Fold Cross-Validation
print("\nPerforming 5-Fold Cross-Validation on log scale...")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2 = cross_val_score(rf, X_clean, y_log, cv=kfold, scoring='r2')
cv_rmse = np.sqrt(-cross_val_score(rf, X_clean, y_log, cv=kfold, 
                                   scoring='neg_mean_squared_error'))

print(f"\n5-Fold Cross-Validation Results (log scale):")
print(f"  CV R²:   {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"  CV RMSE: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")

# Train on ALL data
print(f"\nTraining final model on all {len(X_clean)} samples...")
rf.fit(X_clean, y_log)

# Make predictions (log scale)
y_log_pred = rf.predict(X_clean)

# Back-transform to original scale
y_pred = np.exp(y_log_pred)
y_actual = np.exp(y_log)

# Calculate metrics on original scale
full_r2 = r2_score(y_actual, y_pred)
full_rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

print(f"\nFull Dataset Performance (original scale):")
print(f"  R²:   {full_r2:.4f}")
print(f"  RMSE: {full_rmse:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_clean.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
print(feature_importance.to_string(index=False))

# ============================================================================
# SAVE MODEL
# ============================================================================

models_dir = Path("../Models/Final")
models_dir.mkdir(exist_ok=True, parents=True)

model_path = models_dir / "rf_log_final_model.joblib"
joblib.dump(rf, model_path)

print(f"\nModel saved to: {model_path}")

# Save metadata
import json
metadata = {
    'n_samples': len(X_clean),
    'features': list(X_clean.columns),
    'cv_r2_mean': cv_r2.mean(),
    'cv_r2_std': cv_r2.std(),
    'cv_rmse_mean': cv_rmse.mean(),
    'cv_rmse_std': cv_rmse.std(),
    'full_r2': full_r2,
    'full_rmse': full_rmse,
    'target_transform': 'log',
    'note': 'Model predicts log(Secchi). Use np.exp() to back-transform.'
}

metadata_path = models_dir / "rf_log_model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")

print("\n" + "="*80)
print("COMPLETE!")
print("="*80)
print("\n⚠️  IMPORTANT: This model predicts log(Secchi)!")
print("   To get Secchi, use: np.exp(prediction)")

Loading data...
Original dataset: 1971 rows
Clean dataset: 1971 rows
Features: ['ESTCHL_SUMMED', 'K_PAR_SLOPE', 'XMISS_SUMMED']

TRAINING RANDOM FOREST ON LOG(SECCHI) - 100% DATA

Performing 5-Fold Cross-Validation on log scale...

5-Fold Cross-Validation Results (log scale):
  CV R²:   0.7764 ± 0.0163
  CV RMSE: 0.2368 ± 0.0076

Training final model on all 1971 samples...

Full Dataset Performance (original scale):
  R²:   0.8926
  RMSE: 2.8695

Feature Importance:
      feature  importance
ESTCHL_SUMMED    0.815710
  K_PAR_SLOPE    0.127469
 XMISS_SUMMED    0.056821

Model saved to: ../Models/Final/rf_log_final_model.joblib
Metadata saved to: ../Models/Final/rf_log_model_metadata.json

COMPLETE!

⚠️  IMPORTANT: This model predicts log(Secchi)!
   To get Secchi, use: np.exp(prediction)


In [2]:
import sys
print(sys.executable)
import matplotlib
print(matplotlib.__file__)

/Users/aarti/Downloads/Scripps/Scripps-Light-Attenuation-Capstone-Project/.scripps/bin/python
/Users/aarti/Downloads/Scripps/Scripps-Light-Attenuation-Capstone-Project/.scripps/lib/python3.11/site-packages/matplotlib/__init__.py


In [9]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

In [10]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Load data
df = pd.read_parquet("../Data/Parquet/daylight_satellite_poly_predictions.parquet")

# Load model
rf = joblib.load("../Models/Final/rf_log_final_model.joblib")

# Prepare features
X = df[["ESTCHL_SUMMED", "K_PAR_SLOPE", "XMISS_SUMMED"]]

# Predict log(Secchi)
y_log_pred = rf.predict(X)

# Back-transform to Secchi
df['Predicted_Secchi_RF'] = np.exp(y_log_pred)

# Save
df.to_parquet("../Data/Parquet/daylight_satellite_all_predictions.parquet", index=False)

print("Done! Predictions should now match polynomial regression.")

Done! Predictions should now match polynomial regression.
